# Reproducing Mnemosyne (SIGMOD '25) on Chameleon

**Paper:** *Mnemosyne: Dynamic Workload-Aware BF Tuning via Accurate Statistics in LSM trees*
Zichen Zhu, Yanpeng Wei, Ju Hyoung Mun, Manos Athanassoulis. SIGMOD 2025. https://doi.org/10.1145/3725327

**Original code:** https://github.com/BU-DiSC/Mnemosyne

This notebook reproduces **Figure 4**: the number of unnecessarily accessed data blocks for RocksDB, Monkey and the optimal Bloom filter allocation, while varying the share of empty point queries.

Steps: reserve a bare-metal machine → start a server → connect by SSH → clone the forked repo → run `automate.sh` → show the figure.

# Create lease
1. Choose site
2. Select a project that you are member of
3. Specify a lease name that you are not already using in this site
4. Choose the desired properties (e.g., "compute_skylake") of the baremetal machine
5. Verify that the lease was successful

In [ ]:
import chi
chi.use_site('CHI@TACC')

PROJECT_name = "CHI-261568"   # change this to your own Chameleon project
chi.set("project_name", PROJECT_name)
print(f'Using Project {chi.get("project_name")}')

In [ ]:
from chi import lease

import time
import keystoneauth1, blazarclient
from chi import lease

reservations = []
LEASE_KEY = 'reproduce-sigmod25-mnemosyne'
lease_node_type = "compute_skylake"
try:
    lease_obj=lease.get_lease(LEASE_KEY)
except Exception as e:
    lease_obj=None
if lease_obj!=None:
    print("lease already created")
    print(lease_obj.node_reservations[0]["id"])
else:
    try:
        print(f"Creating lease with name = {LEASE_KEY}...")
        lease.add_fip_reservation(reservations, count=1)
        lease.add_node_reservation(reservations, count=1, node_type=lease_node_type)

        start_date, end_date = lease.lease_duration(hours=24, days=0)

        l = lease.create_lease(
            LEASE_KEY, 
            reservations, 
            start_date=start_date, 
            end_date=end_date
        )
        lease_id = l["id"]
        lease_obj=lease.get_lease(LEASE_KEY)

        print("Waiting for lease to start ...")
        lease_obj.wait()
        print("Lease is now active!")
    except blazarclient.exception.BlazarClientException as e:
        print(f"There is an issue making the reservation. Please check the host calendar.")
        print("https://chi.tacc.chameleoncloud.org/project/leases/calendar/host/")
        print(e)
    except Exception as e:
        print("An unexpected error happened.")
        print(e)

In [ ]:
print(lease_obj.node_reservations[0]["id"])
print(lease_obj)

# Create server and assign ip

In [ ]:
from chi import server
server.list_servers()

image = "CC-Ubuntu24.04"
LEASE_KEY = 'reproduce-sigmod25-mnemosyne'
try:
    s=server.get_server(LEASE_KEY)
except Exception as e:
    s=None

if s!=None:
    print("server already started")
else:
    try:
        s = server.create_server(
            LEASE_KEY, 
            image_name=image,
            reservation_id=lease_obj.node_reservations[0]["id"]
        )
    except Exception as e:
        print("An unexpected error happened.")
        print(e)

    print("Waiting for server to start ...")
    server.wait_for_active(s.id)
    print("Done")

In [ ]:
server.get_server('reproduce-sigmod25-mnemosyne').status

In [ ]:
import chi
from chi import lease

server_id = chi.server.get_server_id(LEASE_KEY)
print(server_id)

floating_ip = lease_obj.get_reserved_floating_ips()
print(floating_ip)
floating_ip = floating_ip[0]
try:
    ip=server.get_floating_ip()
except Exception as e:
    ip=None
if ip != None:
    print("server already get the ip")
else:
    server.associate_floating_ip(server_id, floating_ip_address=floating_ip)
ip = chi.server.get_server(LEASE_KEY).get_floating_ip()
print(ip)

In [ ]:
print(f"Server is accessible at IP address: {ip}")
chi.server.wait_for_tcp(ip, port=22) # Wait for SSH to be ready

# Create SSH connection
All commands are passed through this connection

In [ ]:
import chi
from chi.ssh import Remote
import paramiko

key_path = "/work/.ssh/id_rsa"

# Load your private key
pkey = paramiko.RSAKey.from_private_key_file(key_path)

conn = Remote(
    ip=ip,
    user="cc",
    connect_kwargs={
        "pkey": pkey,
        "key_filename": [],
        "allow_agent": False,
        "look_for_keys": False,
    },
)

conn.run("hostname")

# Check properties of the newly created instance

In [ ]:
conn.run('free -h')
conn.run('df -h')
conn.run('nproc')

# Clone the repository for experiment reproducibility (forked from the original)

In [ ]:
conn.run('git clone https://github.com/aroma058/trovi_reproduce_mnemosyne-sigmod25-fork.git')

In [ ]:
conn.run('ls -al trovi_reproduce_mnemosyne-sigmod25-fork')

# Run the script that reproduces Figure 4
`automate.sh` does all of this:
1. Installs dependencies
2. Downloads the code the paper needs (RocksDB fork, benchmark, workload generator)
3. Compiles everything
4. Makes an 80 GB RAM disk for the database
5. Runs the authors' `fig4.sh`, which saves `exp-figures/Fig4.png`

This takes several hours, so it runs in the background. The output goes to `run.log`.

In [ ]:
conn.run('chmod 777 trovi_reproduce_mnemosyne-sigmod25-fork/automate.sh')
conn.run('cd trovi_reproduce_mnemosyne-sigmod25-fork && nohup ./automate.sh > run.log 2>&1 &', pty=False)
print("Started. Use the next cell to check progress.")

# Check progress
Run this cell again any time. When it is finished, the last line says **Done. Figure saved at exp-figures/Fig4.png**.

In [ ]:
conn.run('tail -n 15 trovi_reproduce_mnemosyne-sigmod25-fork/run.log')

# Get the figure so that it can be viewed in the UI
The script has generated the figure named Fig4.png

In [ ]:
conn.get(
    'trovi_reproduce_mnemosyne-sigmod25-fork/exp-figures/Fig4.png',
    'Fig4.png'
)

In [ ]:
from IPython.display import Image, display
display(Image(filename='Fig4.png'))

# Compare with Figure 4 from the authors' repository

In [ ]:
import urllib.request
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/BU-DiSC/Mnemosyne/main/exp-figures/Fig4.png',
    'Fig4_original.png'
)
display(Image(filename='Fig4_original.png'))

# Clean up
Delete the server and the lease when you are done, so the machine is free for others.

In [ ]:
server.delete_server(server_id)
lease.delete_lease(lease_obj.id)